**Preprocess & fine-tune transformer-based models**

**1. Understanding BERT and XLM-RoBERTa**

- BERT stands for Bidirectional Encoder Representations from Transformers.
It is an encoder-only transformer model that reads text in both directions at the same time: left-to-right and right-to-left. This allows BERT to understand the context of a word based on the words before and after it. Commonly used for text classification, sentiment analysis, question answering, named entity recognition, and sentence similarity tasks.

- XLM-RoBERTa is a multilingual transformer model based on RoBERTa.
It was trained on text from many languages, so it works well for multilingual NLP tasks. Better for multilingual classification because it was specifically trained on a large multilingual corpus.

**2. Tokenizing Text**

Transformer models cannot read raw text directly. First, text must be converted into tokens and then into numerical IDs.

BERT may split words into subwords using WordPiece tokenization.

XLM-RoBERTa uses SentencePiece tokenization, which works better across many languages.

For English-only classification, bert-base-uncased is usually a good starting point.

For multilingual datasets, xlm-roberta-base is usually a better choice.

In [4]:
from transformers import BertTokenizer, XLMRobertaTokenizer

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
xlmr_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [20]:
text = "Transformers are amazing!"

encoded = tokenizer(text)

print(encoded)

{'input_ids': [101, 19081, 2024, 6429, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}


**3. Preparing Input Data for the Model**

In [21]:
# Padding + Truncation
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

text = "Transformers are amazing!"

encoded = tokenizer(
    text,
    max_length=10,
    padding="max_length",
    truncation=True,
    return_attention_mask=True
)

print(encoded)
print(tokenizer.special_tokens_map)
print(tokenizer.vocab_size)

{'input_ids': [101, 19081, 2024, 6429, 999, 102, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 0, 0, 0, 0]}
{'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
30522


In [17]:
xlmr = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")
print(xlmr.special_tokens_map)

{'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}


**4. Loading and Exploring the Dataset**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -o "/content/drive/MyDrive/Basics of BERT and XLM-RoBERTa - PyTorch - 2.zip" -d /content/data
!unzip -o "/content/data/Basics of BERT and XLM-RoBERTa - PyTorch/train.csv.zip" -d /content/data
!unzip -o "/content/data/Basics of BERT and XLM-RoBERTa - PyTorch/test.csv.zip" -d /content/data

Mounted at /content/drive
Archive:  /content/drive/MyDrive/Basics of BERT and XLM-RoBERTa - PyTorch - 2.zip
  inflating: /content/data/Basics of BERT and XLM-RoBERTa - PyTorch/sample_submission.csv  
 extracting: /content/data/Basics of BERT and XLM-RoBERTa - PyTorch/test.csv.zip  
 extracting: /content/data/Basics of BERT and XLM-RoBERTa - PyTorch/train.csv.zip  
Archive:  /content/data/Basics of BERT and XLM-RoBERTa - PyTorch/train.csv.zip
  inflating: /content/data/train.csv  
Archive:  /content/data/Basics of BERT and XLM-RoBERTa - PyTorch/test.csv.zip
  inflating: /content/data/test.csv  


In [2]:
import pandas as pd

train_df = pd.read_csv('/content/data/train.csv')
test_df = pd.read_csv('/content/data/test.csv')

train_df.head()

,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
2,3931fbe82a,Des petites choses comme celles-là font une di...,J'essayais d'accomplir quelque chose.,fr,French,0
3,5622f0c60b,you know they can't really defend themselves l...,They can't defend themselves because of their ...,en,English,0
4,86aaa48b45,ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...,เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร,th,Thai,1


In [26]:
train_df = train_df[train_df['language'] == 'English']
train_df.head()


,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
3,5622f0c60b,you know they can't really defend themselves l...,They can't defend themselves because of their ...,en,English,0
7,fdcd1bd867,From Cockpit Country to St. Ann's Bay,From St. Ann's Bay to Cockpit Country.,en,English,2
8,7cfb3d272c,"Look, it's your skin, but you're going to be i...",The boss will fire you if he sees you slacking...,en,English,1


In [27]:
train_df.shape

(6870, 6)

**5. Creating Cross-Validation Folds**

In [32]:
from sklearn.model_selection import StratifiedKFold

premises = train_df["premise"]
labels = train_df["label"]

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

folds = []

for train_idx, val_idx in kf.split(premises, labels):
    train_data = train_df.iloc[train_idx]
    val_data = train_df.iloc[val_idx]

    folds.append((train_data, val_data))

print(len(folds))

5
